# SafeChain Pipeline — Integration Test

End-to-end test with a single case and a specific question:

> **"How frequent were positive events (e.g. limit increases) in the last 18 months?"**

**Run this notebook in the deployment environment where `safechain` is installed.**

---
## 0. Setup

In [ ]:
import os, sys, json

# Resolve project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'notebooks':
    PROJECT_ROOT = os.getcwd()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'simulated')
PROFILE_DIR = os.path.join(PROJECT_ROOT, 'config', 'data_profiles')
PILLAR_DIR = os.path.join(PROJECT_ROOT, 'config', 'pillars')

# ══════════════════════════════════════════════════════════════
# CONFIG — change these for your environment
# ══════════════════════════════════════════════════════════════
SAFECHAIN_MODEL = "gpt-4.1"
CASE_ID = "CASE-00001"
PILLAR = "credit_risk"
QUESTION = "How frequent were positive events (e.g. limit increases) in the last 18 months?"
# ══════════════════════════════════════════════════════════════

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir exists: {os.path.isdir(DATA_DIR)}')
print(f'Profile dir exists: {os.path.isdir(PROFILE_DIR)}')
print(f'Case: {CASE_ID}')
print(f'Question: {QUESTION}')

---
## 1. Load Data Layer & Set Case

In [ ]:
from data.gateway import SimulatedDataGateway
from data.catalog import DataCatalog
from tools.data_tools import init_tools, list_available_tables, get_table_schema, query_table

gw = SimulatedDataGateway.from_case_folders(DATA_DIR)
catalog = DataCatalog(profile_dir=PROFILE_DIR)
init_tools(gw, catalog)

gw.set_case(CASE_ID)
print(f'Case {CASE_ID} loaded')
print(f'Tables: {gw.list_tables()}')

---
## 2. Preview Case Data (relevant tables)

Before running the pipeline, let's see what data the specialists will work with.

In [ ]:
# Bureau — check positive_events field and scores
print('=== Bureau Full (first 2 rows) ===')
print(query_table('bureau_full', limit=2))

In [ ]:
# Model scores — positive_events is one of the 264 model variables
# query_table truncates wide tables, so query the gateway directly for specific fields
print('=== Model Scores — positive_events field ===')
rows = gw.query('model_scores', limit=5)
if rows:
    for i, row in enumerate(rows):
        print(f'Row {i}: positive_events={row.get("positive_events", "N/A")}')
else:
    print('No model_scores data for this case')

In [ ]:
# Cross-BU — card tenure and account status (limit increases relate to card history)
print('=== Cross-BU Cards ===')
print(query_table('cross_bu'))

In [ ]:
# Customer tenure
print('=== Customer Tenure ===')
print(query_table('cust_tenure'))

---
## 3. Connect SafeChain & Create Adapter

In [ ]:
from gateway.safechain_adapter import SafeChainAdapter

try:
    from safechain.lcel import model as safechain_model
    llm = safechain_model(SAFECHAIN_MODEL)
    print(f'SafeChain LLM created: model={SAFECHAIN_MODEL}')
except ImportError:
    print('SafeChain not available — falling back to OpenAI adapter')
    from gateway.openai_adapter import OpenAIAdapter
    llm = None

if llm is not None:
    adapter = SafeChainAdapter(llm=llm, model_name=SAFECHAIN_MODEL)
else:
    adapter = OpenAIAdapter(model='gpt-4.1')

print(f'Adapter: {type(adapter).__name__}')

---
## 4. Quick Smoke Test — Raw LLM Call

In [ ]:
result = adapter.chat_turn([
    {"role": "system", "content": "You are a credit risk analyst."},
    {"role": "user", "content": "What are positive events in credit risk analysis? Answer in 2 sentences."},
])
print(result)

---
## 5. Initialize Pipeline Components

In [ ]:
from gateway.firewall_stack import FirewallStack
from log.event_logger import EventLogger
from agents.session_registry import SessionRegistry
from agents.base_agent import BaseSpecialistAgent
from agents.general_specialist import GeneralSpecialist
from orchestrator.orchestrator import Orchestrator
from orchestrator.team import TeamConstructor
from orchestrator.chat_agent import ChatAgent
from config.pillar_loader import PillarLoader
from skills.domain.loader import load_domain_skill, list_domain_skills

logger = EventLogger(
    session_id=f'test-{CASE_ID}',
    log_dir=os.path.join(PROJECT_ROOT, 'logs'),
)
pillar_loader = PillarLoader(pillar_dir=PILLAR_DIR)
registry = SessionRegistry()

logger.log('session_start', {'case_id': CASE_ID, 'pillar': PILLAR, 'question': QUESTION})
print('Pipeline components initialized')
print(f'Available domain skills: {list_domain_skills()}')

---
## 6. Team Construction

The orchestrator selects which specialists are relevant to answer the question.

In [ ]:
team_firewall = FirewallStack(adapter=adapter, logger=logger)
team_constructor = TeamConstructor(firewall=team_firewall, logger=logger)

logger.set_trace('q-001')
logger.log('orchestrator_dispatch', {'question': QUESTION, 'pillar': PILLAR})

selected = team_constructor.select_specialists(
    question=QUESTION,
    pillar=PILLAR,
    available_specialists=list_domain_skills(),
    active_specialists=registry.list_active(),
)

print(f'Specialists selected: {selected}')
logger.log('orchestrator_dispatch', {'specialists': selected})

---
## 7. Run Selected Specialists

Each specialist runs: Data Request → Synthesize → Answer (chat mode).

In [ ]:
specialist_outputs = {}

for domain in selected:
    print(f'\n{"="*60}')
    print(f'Running specialist: {domain}')
    print(f'{"="*60}')
    
    skill = load_domain_skill(domain)
    if skill is None:
        print(f'  SKIP — domain skill not found')
        continue
    
    spec_config = pillar_loader.get_specialist_config(PILLAR, domain) or {}
    spec_firewall = FirewallStack(adapter=adapter, logger=logger)
    
    agent = registry.get_or_create(
        domain=domain, pillar=PILLAR,
        domain_skill=skill, pillar_yaml=spec_config,
        firewall=spec_firewall, logger=logger,
    )
    
    output = agent.run(QUESTION, mode='chat')
    specialist_outputs[domain] = output
    
    print(f'\n  Findings: {output.findings[:300]}')
    print(f'  Evidence: {output.evidence[:3]}')
    print(f'  Implications: {output.implications[:3]}')
    print(f'  Data gaps: {output.data_gaps}')

print(f'\n{"="*60}')
print(f'Specialist outputs collected: {list(specialist_outputs.keys())}')

---
## 8. General Specialist — Compare

Examines specialist outputs pairwise for contradictions, then self-resolves.

In [ ]:
compare_firewall = FirewallStack(adapter=adapter, logger=logger)
general = GeneralSpecialist(firewall=compare_firewall, logger=logger)

review_report = general.compare(specialist_outputs, QUESTION)

print(f'Resolved contradictions: {len(review_report.resolved)}')
for r in review_report.resolved:
    print(f'\n  Pair: {r.pair}')
    print(f'  Contradiction: {r.contradiction}')
    print(f'  Question raised: {r.question_raised}')
    print(f'  Answer: {r.answer[:200]}')
    print(f'  Conclusion: {r.conclusion}')

print(f'\nOpen conflicts: {len(review_report.open_conflicts)}')
for c in review_report.open_conflicts:
    print(f'  Pair: {c.pair}')
    print(f'  Why unresolved: {c.reason_unresolved}')

print(f'\nCross-domain insights: {review_report.cross_domain_insights}')

---
## 9. Orchestrator Synthesize

Merges specialist outputs + review report into a single coherent answer.
Evaluates absence-as-signal, flags open conflicts.

In [ ]:
synth_firewall = FirewallStack(adapter=adapter, logger=logger)
orchestrator = Orchestrator(
    firewall=synth_firewall,
    logger=logger,
    registry=registry,
    pillar=PILLAR,
)

final = orchestrator.synthesize(
    specialist_outputs=specialist_outputs,
    review_report=review_report,
    question=QUESTION,
    mode='chat',
)

print('=== Synthesized Answer ===')
print(final.answer)
print(f'\nResolved contradictions: {len(final.resolved_contradictions)}')
print(f'Open conflicts: {len(final.open_conflicts)}')
print(f'Data gaps: {len(final.data_gaps)}')
for g in final.data_gaps:
    print(f'  {g.specialist}: {g.missing_data}')
    print(f'    Interpretation: {g.absence_interpretation}')
    print(f'    Is signal: {g.is_signal}')
print(f'Blocked steps: {len(final.blocked_steps)}')
print(f'Specialists consulted: {final.specialists_consulted}')

---
## 10. Format for Reviewer

In [ ]:
chat_agent = ChatAgent(firewall=synth_firewall, logger=logger)
formatted = chat_agent.format_for_reviewer(final)

print('='*60)
print(f'QUESTION: {QUESTION}')
print(f'CASE: {CASE_ID} | PILLAR: {PILLAR}')
print('='*60)
print(formatted)
print('='*60)

logger.log('final_output', {
    'question': QUESTION,
    'specialists': final.specialists_consulted,
    'open_conflicts': len(final.open_conflicts),
    'data_gaps': len(final.data_gaps),
})
logger.clear_trace()

---
## 11. Inspect Event Log

Trace how the answer was generated — every inter-agent communication is logged.

In [ ]:
log_path = os.path.join(PROJECT_ROOT, 'logs', f'test-{CASE_ID}.jsonl')

if os.path.exists(log_path):
    with open(log_path) as f:
        events = [json.loads(line) for line in f]
    
    print(f'Total events: {len(events)}')
    print()
    
    # Event type summary
    from collections import Counter
    type_counts = Counter(e['event'] for e in events)
    print('Event type counts:')
    for event_type, count in type_counts.most_common():
        print(f'  {event_type}: {count}')
else:
    print(f'No log file at {log_path}')

In [ ]:
# Full trace for question q-001
if os.path.exists(log_path):
    trace_events = [e for e in events if e.get('trace_id') == 'q-001']
    print(f'Events in trace q-001: {len(trace_events)}')
    print()
    for e in trace_events:
        ts = e['timestamp'].split('T')[1][:8]
        event_type = e['event']
        # Build a compact summary line
        extras = {k: v for k, v in e.items() if k not in ('timestamp', 'session_id', 'trace_id', 'event')}
        extra_str = json.dumps(extras) if extras else ''
        if len(extra_str) > 150:
            extra_str = extra_str[:150] + '...'
        print(f'  [{ts}] {event_type}  {extra_str}')

---
## 12. Active Specialist Registry

Shows which specialists are warm and can be reused for follow-up questions.

In [ ]:
active = registry.list_active()
print(f'Active specialists: {len(active)}')
for a in active:
    print(f'  {a["domain"]} ({a["pillar"]}): {a["questions_answered"]} questions answered')
    if a['summary_preview']:
        print(f'    Summary: {a["summary_preview"][:200]}...')

---
## 13. (Optional) Ask a Follow-Up Question

Specialists are warm — their rolling summary from the first question carries forward.

In [ ]:
FOLLOW_UP = "Are the positive events consistent with the bureau scores?"

logger.set_trace('q-002')
logger.log('orchestrator_dispatch', {'question': FOLLOW_UP, 'pillar': PILLAR})

# Reuse the same specialists — they already have context
followup_outputs = {}
for domain in selected:
    skill = load_domain_skill(domain)
    spec_config = pillar_loader.get_specialist_config(PILLAR, domain) or {}
    spec_firewall = FirewallStack(adapter=adapter, logger=logger)
    
    # get_or_create returns the SAME instance — rolling summary is still there
    agent = registry.get_or_create(
        domain=domain, pillar=PILLAR,
        domain_skill=skill, pillar_yaml=spec_config,
        firewall=spec_firewall, logger=logger,
    )
    
    output = agent.run(FOLLOW_UP, mode='chat')
    followup_outputs[domain] = output
    print(f'{domain}: {output.findings[:200]}')
    print()

# Compare + Synthesize
review2 = general.compare(followup_outputs, FOLLOW_UP)
final2 = orchestrator.synthesize(followup_outputs, review2, FOLLOW_UP, 'chat')

print('='*60)
print(f'FOLLOW-UP: {FOLLOW_UP}')
print('='*60)
print(chat_agent.format_for_reviewer(final2))

logger.clear_trace()

In [ ]:
# Check rolling summary grew with both questions
for a in registry.list_active():
    print(f'{a["domain"]}: {a["questions_answered"]} questions answered')

In [ ]:
logger.log('session_end', {})
print('Session complete.')